# CP4 · Notebook 05 — Evaluación in-distribution vs OOD

Análisis cuantitativo + qualitativo del policy entrenado. ~5 min.

In [ ]:
import numpy as np, json
from pathlib import Path
import torch, torch.nn as nn
import matplotlib.pyplot as plt

OUT = Path('../outputs')
data = np.load('../datasets/cp4-highway-bc.npz')

class PilotNetMini(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 24, 5, stride=2), nn.ReLU(),
            nn.Conv2d(24, 36, 5, stride=2), nn.ReLU(),
            nn.Conv2d(36, 48, 3, stride=2), nn.ReLU(),
            nn.Conv2d(48, 64, 3), nn.ReLU(),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 6 * 6, 100), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(100, 1), nn.Tanh())
    def forward(self, x): return self.head(self.conv(x)).squeeze(-1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = PilotNetMini().to(device)
model.load_state_dict(torch.load(OUT / '04_pilotnet.pt', map_location=device))
model.eval()
print('modelo cargado')

## 1. Predicciones sobre val_in y val_ood

In [ ]:
def predict(obs_split):
    obs = obs_split.astype(np.float32) / 255.0
    obs_t = torch.from_numpy(obs.transpose(0, 3, 1, 2)).to(device)
    with torch.no_grad():
        return model(obs_t).cpu().numpy()

pred_in = predict(data['val_in_obs'])
pred_ood = predict(data['val_ood_obs'])
true_in = data['val_in_actions']
true_ood = data['val_ood_actions']

mse_in = float(((pred_in - true_in) ** 2).mean())
mse_ood = float(((pred_ood - true_ood) ** 2).mean())
mae_in = float(np.abs(pred_in - true_in).mean())
mae_ood = float(np.abs(pred_ood - true_ood).mean())

print(f'Val in-dist:  MSE={mse_in:.4f}  MAE={mae_in:.4f}')
print(f'Val OOD:      MSE={mse_ood:.4f}  MAE={mae_ood:.4f}')
print(f'Gap relativo: {(mse_ood-mse_in)/mse_in*100:+.0f}%')

## 2. Scatter predicción vs ground truth

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (pred, true, title, color) in zip(axes, [
        (pred_in,  true_in,  'val in-dist',  '#4a6fa5'),
        (pred_ood, true_ood, 'val OOD',      '#DA4544')]):
    ax.scatter(true, pred, alpha=0.4, s=12, color=color)
    ax.plot([-1, 1], [-1, 1], 'k--', alpha=0.5, label='y=x perfecto')
    ax.set_xlabel('steering ground truth')
    ax.set_ylabel('steering predicho')
    ax.set_title(title)
    ax.set_xlim(-1.1, 1.1); ax.set_ylim(-1.1, 1.1); ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.savefig(OUT / '05_scatter.png', dpi=100, bbox_inches='tight'); plt.show()

**Patrón típico**:
- En `val in-dist` los puntos se concentran cerca de la diagonal — el modelo "aprende" la regresión.
- En `val OOD` la nube se aplana hacia y=0 — el modelo **regresa hacia el sesgo del training** (steering ≈ 0) cuando le muestran algo distinto.

Esto es el comportamiento de un modelo que **memoriza la distribución de training y falla en OOD**, no porque sea malo sino porque no aprendió nada generalizable.

## 3. Predicciones cualitativas — 4 ejemplos por split

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for row, (split, color) in enumerate([('val_in', '#4a6fa5'), ('val_ood', '#DA4544')]):
    obs_split = data[f'{split}_obs']
    act_split = data[f'{split}_actions']
    pred_split = pred_in if split == 'val_in' else pred_ood
    # 4 ejemplos donde la predicción difiere más
    diff = np.abs(pred_split - act_split)
    worst = np.argsort(diff)[-4:][::-1]
    for col, i in enumerate(worst):
        ax = axes[row, col]
        ax.imshow(obs_split[i, :, :, 2], cmap='gray')
        ax.set_title(f'{split}[{i}]\nGT={act_split[i]:+.2f}  pred={pred_split[i]:+.2f}\nerror={diff[i]:.2f}',
                     fontsize=9, color=color)
        ax.axis('off')
plt.tight_layout(); plt.savefig(OUT / '05_worst_predictions.png', dpi=100, bbox_inches='tight'); plt.show()

## 4. Guardar resumen

In [ ]:
summary = {
    'val_in':  {'mse': mse_in,  'mae': mae_in},
    'val_ood': {'mse': mse_ood, 'mae': mae_ood},
    'gap_relative_pct': (mse_ood - mse_in) / mse_in * 100,
}
with open(OUT / '05_eval_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))
print('\nVe a 06_compounding_error.ipynb — vamos a desplegar el policy en el entorno.')